[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madhudream/rag_simple/blob/main/colab/05_contextual_retrieval.ipynb)

# 05 — Contextual retrieval: stamp every chunk with what it's about

Companion notebook to blog post **05 (Contextual retrieval)** — Anthropic's technique,
the course's biggest single quality jump.

The return-address metaphor (post 01's index cards, fixed):
1. A card **ripped from its book forgets the book** ("Refunds in 5 days" — of WHAT?)
2. At filing time you still hold the book — **an LLM writes a one-line stamp** per chunk
3. The stamp **moves the card on the meaning-map** — twins stop being twins
4. **Stamp once, benefit forever** — index-time cost, zero query-time cost (mirror of 04)

Needs an OpenAI API key (writing the stamps). Stamps vary run to run; the flip and the
widened margins are what reproduce.

In [ ]:
%pip install -q fastembed numpy openai

In [ ]:
import os

def load_key():
    try:                                      # Colab: add OPENAI_API_KEY in the Secrets
        from google.colab import userdata     # sidebar (key icon) and enable notebook access
        return userdata.get("OPENAI_API_KEY")
    except Exception:                         # local Jupyter fallback
        import getpass
        return os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_key()

## The corpus — two pages whose chunks are twins

Paragraph-chunk (post 01's winning cut) and the page titles fall away. The two refund
paragraphs become twins: same vocabulary, same shape, nothing says which service.

In [ ]:
PAGES = {
"boarding": """Sunnyvale Pet Care — Boarding Services

The boarding facility is open seven days a week. It closes at 7 pm on weekdays and 5 pm on weekends.

Guests staying longer than three nights receive a complimentary bath before pickup.

Cancellations must be made 48 hours in advance. Refunds are issued within 5 business days.""",

"grooming": """Sunnyvale Pet Care — Grooming Salon

Appointments must be booked at least 48 hours in advance. Walk-ins are not available.

The full package includes a bath, haircut, nail trim, and ear cleaning.

Cancellations made with less than 24 hours notice incur a fee. Refunds are issued within 10 business days.""",
}

chunks, meta = [], []
for name, page in PAGES.items():
    for p in page.split("\n\n")[1:]:      # [1:] drops the page title — chunks lose their page
        chunks.append(p)
        meta.append(name)

for i, (c, m) in enumerate(zip(chunks, meta)):
    print(f"  chunk {i} [{m:8}] {c[:70]}")

In [ ]:
import numpy as np
from fastembed import TextEmbedding
from openai import OpenAI

emb_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
client = OpenAI()

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def llm(prompt):
    r = client.chat.completions.create(model="gpt-5.4-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip()

def search(query, embs, top_k=3):
    q = list(emb_model.embed([query]))[0]
    return sorted(((cosine(q, e), i) for i, e in enumerate(embs)), reverse=True)[:top_k]

plain_embs = list(emb_model.embed(chunks))

## The villain — the retriever picks the wrong twin

Watch three queries: one outright WRONG (grooming refund → boarding chunk), one
coin-flip win (0.02 margin = noise), one with the wrong page's chunk lurking.

In [ ]:
QUERIES = [
    ("How long do grooming refunds take?", "grooming"),
    ("When do I get my money back for a cancelled boarding stay?", "boarding"),
    ("Does boarding include a bath?", "boarding"),
]
for q, gold in QUERIES:
    print(f"\nQ: {q}   (right page: {gold})")
    for s, i in search(q, plain_embs):
        mark = "  <-- WRONG page" if meta[i] != gold else ""
        print(f"  {s:.3f}  [{meta[i]:8}] {chunks[i][:58]}{mark}")

## The stamp — Anthropic's situate-this-chunk prompt

At index time we still hold the full document. One LLM call per chunk writes the
return address; the STAMPED text (context + original) is what gets embedded.

In [ ]:
CTX_PROMPT = """<document>
{document}
</document>

Here is the chunk we want to situate within the whole document:
<chunk>
{chunk}
</chunk>

Please give a short succinct context to situate this chunk within the overall
document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else."""

ctx_chunks = []
for c, m in zip(chunks, meta):
    ctx = llm(CTX_PROMPT.format(document=PAGES[m], chunk=c))
    ctx_chunks.append(f"{ctx}\n{c}")       # stamp on top, original below
    print(f"  [{m:8}] {ctx}")

ctx_embs = list(emb_model.embed(ctx_chunks))   # re-embed — stamped text is new text

## Same queries, stamped chunks — read the margins

Expect: the wrong twin FLIPS (decisively), the coin-flip becomes a landslide, the
lurker gets pushed out. Margins are the real product — thin margins are where
retrieval breaks under paraphrase and corpus growth.

In [ ]:
for q, gold in QUERIES:
    print(f"\nQ: {q}   (right page: {gold})")
    for s, i in search(q, ctx_embs):
        mark = "  <-- WRONG page" if meta[i] != gold else ""
        print(f"  {s:.3f}  [{meta[i]:8}] {ctx_chunks[i].splitlines()[0][:58]}{mark}")

## Scoreboard — measured every step

In [ ]:
GOLDEN = QUERIES + [
    ("What time does boarding close on weekends?", "boarding"),
    ("Is there a fee for late grooming cancellation?", "grooming"),
    ("What does the full grooming package include?", "grooming"),
    ("How far ahead must grooming appointments be booked?", "grooming"),
    ("Do long stays come with a free bath?", "boarding"),
]

def hit1(embs):
    hits, misses = 0, []
    for q, gold in GOLDEN:
        s, i = search(q, embs, 1)[0]
        hits += meta[i] == gold
        if meta[i] != gold:
            misses.append(q[:45])
    return hits, misses

ph, pm = hit1(plain_embs)
ch, cm = hit1(ctx_embs)
print(f"plain chunks          hit@1 = {ph}/{len(GOLDEN)}   misses: {pm}")
print(f"contextualized chunks hit@1 = {ch}/{len(GOLDEN)}   misses: {cm}")

## PRODUCTION — the per-DOCUMENT compromise

The cost lives at index time: 1 LLM call per chunk, once. At course scale that was
38,000 calls — which saturated rate limits and exhausted the API quota mid-run. The
shipped fix: **one context per document** (27× fewer calls), prepended to all its
chunks. Slightly blunter stamps, wildly cheaper. Course numbers (with this compromise,
full corpus, 50 golden Qs): recall 0.78→0.83, precision 0.852→0.945, faithfulness
0.909→**0.968** (best in course), latency 6.50→2.56 s — because multi-query (04) was
DROPPED: stamps already did its job at index time; ctx+multi together measured WORSE
than ctx alone.

In [ ]:
DOC_CTX_PROMPT = """Summarize what this document is about in one sentence, naming the
service it covers. Answer with the sentence only.

{document}"""

doc_ctx = {name: llm(DOC_CTX_PROMPT.format(document=page)) for name, page in PAGES.items()}
for name, ctx in doc_ctx.items():
    print(f"  [{name:8}] {ctx}")

perdoc_chunks = [f"{doc_ctx[m]}\n{c}" for c, m in zip(chunks, meta)]
perdoc_embs = list(emb_model.embed(perdoc_chunks))

dh, dm = hit1(perdoc_embs)
print(f"\nper-chunk stamps  hit@1 = {ch}/{len(GOLDEN)}   ({len(chunks)} LLM calls)")
print(f"per-doc stamps    hit@1 = {dh}/{len(GOLDEN)}   ({len(PAGES)} LLM calls)   misses: {dm}")

## Recap — the return-address rules

1. **A card forgets its book** → twin chunks, wrong one retrieved
2. **Stamp it while you hold the book** → LLM situates each chunk; stamp prepended, embedded
3. **The stamp moves the card** → flips + landslide margins; hit@1 up
4. **Stamp once, benefit forever** → index-time batch job (budget it!); zero query-time
   cost; retired multi-query at course scale

**Next: semantic cache (06)** — attack cost and speed: recognize that two differently
phrased questions are the same question, and answer the second in 0.2 s for free.